In [1]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\train"
VAL_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\val"
TEST_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\test"

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_generator.num_classes

Found 2351 images belonging to 3 classes.
Found 506 images belonging to 3 classes.
Found 505 images belonging to 3 classes.


In [3]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

In [4]:
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,591,811 (9.89 MB)

 Trainable params: 331,267 (1.26 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [5]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "ResNet50_transfer_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

In [6]:
history_head = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5495 - loss: 1.0988
Epoch 1: val_accuracy improved from -inf to 0.78854, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 159s 2s/step - accuracy: 0.5508 - loss: 1.0965 - val_accuracy: 0.7885 - val_loss: 0.5181 - learning_rate: 1.0000e-04
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7294 - loss: 0.6645
Epoch 2: val_accuracy improved from 0.78854 to 0.81818, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - accuracy: 0.7297 - loss: 0.6639 - val_accuracy: 0.8182 - val_loss: 0.4066 - learning_rate: 1.0000e-04
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7690 - loss: 0.6127
Epoch 3: val_accuracy improved from 0.81818 to 0.83992, saving model to ResNet50_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 162s 2s/step - accuracy: 0.7693 - loss: 0.6119 - val_accuracy: 0.8399 - val_loss: 0.3629 - learning_rate: 1.0000e-04
Epoch 4/50
74/74 ━━

In [7]:
import json

with open("MobileNetV2_head_history_LWDCD.json", "w") as f:
    json.dump(history_head.history, f)

In [8]:
base_model.trainable = True

for layer in base_model.layers[:120]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8577 - loss: 0.3622
Epoch 1: val_accuracy did not improve from 0.94862
74/74 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.8577 - loss: 0.3622 - val_accuracy: 0.9348 - val_loss: 0.1733 - learning_rate: 1.0000e-05
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8725 - loss: 0.3171
Epoch 2: val_accuracy did not improve from 0.94862
74/74 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - accuracy: 0.8724 - loss: 0.3174 - val_accuracy: 0.9130 - val_loss: 0.2269 - learning_rate: 1.0000e-05
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8813 - loss: 0.3054
Epoch 3: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.

Epoch 3: val_accuracy did not improve from 0.94862
74/74 ━━━━━━━━━━━━━━━━━━━━ 119s 2s/step - accuracy: 0.8813 - loss: 0.3054 - val_accuracy: 0.8972 - val_loss: 0.2555 - learning_rate: 1.0000e-05
Epoch 4/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8954 - loss: 0.2719
Epoch

In [9]:
with open("MobileNetV2_finetune_history_LWDCD.json", "w") as f:
    json.dump(history_fine.history, f)

In [10]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.4f}")

16/16 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.9264 - loss: 0.2317
Test accuracy: 0.9366


In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_probs = model.predict(test_generator)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

16/16 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step
                  precision    recall  f1-score   support

   Healthy Wheat       0.95      0.93      0.94       175
       Leaf Rust       0.98      0.93      0.95       190
Wheat Loose Smut       0.87      0.96      0.91       140

        accuracy                           0.94       505
       macro avg       0.93      0.94      0.94       505
    weighted avg       0.94      0.94      0.94       505

[[162   1  12]
 [  5 176   9]
 [  3   2 135]]


In [12]:
model.save("MobileNetV2_finetuned_LWDCD.keras")

In [13]:
model.save("MobileNetV2_head_LWDCD.keras")